<a href="https://colab.research.google.com/github/LiaIssakov/Portfolio_2025/blob/main/SVD_low_rank_approximation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Low rank approximation with SVD

The idea is to abuse SVD (and in particular the information contained in the singular values) to  approximately represent a matrix $M$ as a product of 3 significantly smaller matrices.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

import ipywidgets as widgets

plt.rcParams['figure.figsize'] = (12.0, 12.0)

In [ ]:
# don't look at me
from mpl_toolkits.axes_grid1 import make_axes_locatable

def plot_images_in_a_row(ims):
  'Takes a list/array of images and plots them spaced horizontally.'
  fig, axs = plt.subplots(1, len(ims), figsize = (20,5))
  for im, ax in zip(ims, axs.ravel()):
    r = ax.imshow(im, cmap = 'gray')
    divider = make_axes_locatable(ax)
    cax = divider.append_axes('right', size='5%', pad=0.05)
    fig.colorbar(r, cax=cax, orientation='vertical')
    ax.axis('off')

# 0) Warmup: multiplication table

Let's think about the multiplication table, say, 10x10.

Viewed as a matrix $M$, what is its rank?

Can you think of two matrices $A,B$ whose product would be $M$? (They may we weird matrices...)

In [ ]:
u,v = np.linspace(1,10,10), np.linspace(1,10,10)

In [ ]:
v@u # these are considered vectors in numpy, so just a number

In [ ]:
v.T@u # this doesn't help!

In [ ]:
assert False

In [ ]:
v.reshape(-1, 1) @ u.reshape(1, -1) # treating them as nx1 and 1xn matrices helps!

In [ ]:
M

In [ ]:
np.linalg.matrix_rank(M)

## Task: Let's try SVD



In [ ]:
U, S, Vt = np.linalg.svd(M)
S = np.diag(S) # annoying: the original S is a vector, we turned it into a matrix

In [ ]:
U@S@Vt

In [ ]:
U.shape

# We can check the vectors corresponding to the dominant singular value

In [ ]:
# multiply: dominant singular value and corresponding (singular) vectors
M1 = S[0,0] * np.outer(U[:, 0], Vt[0])

In [ ]:
M1

In [ ]:
S[0] # extra: where does this number come from?

In [ ]:
# Just a convenient function if you want to play with images of arbitrary size
from scipy.linalg import diagsvd

def nice_general_svd(R):
  '''This function just makes sure SVD will work for matrices (images) of
  arbitrary size. In particular, diagsvd adds padding/zeros to S.'''
  U,S,Vt = np.linalg.svd(R)
  S = diagsvd(S, *R.shape)

  return U,S,Vt

# 1) Serious stuff: images

> We will convert to grayscale -- but you can just apply the below method to each RGB channel!

In [ ]:
#@title
# IGNORE: This just loads a nice image
f = '/content/Dsg_UF_Century_Tower_20050507.jpg'

try:
  im = Image.open(f)
except:
  !wget https://people.clas.ufl.edu/hwagner/files/Dsg_UF_Century_Tower_20050507.jpg

im_orig = np.asarray(Image.open(f))[:500, :500]

In [ ]:
plt.imshow(im_orig)

In [ ]:
im = np.array(im_orig).mean(axis=-1)/255. # make things between 0 and 1 as usual

In [ ]:
print(f"SIZE of image/matrix: {im.shape} RANK of this matrix: {np.linalg.matrix_rank(im)}")

In [ ]:
plt.imshow(im, cmap='gray')

## Simple case for now: black and white image

In [ ]:
im_bw = (im < 0.7)/1.

In [ ]:
plt.imshow(im_bw, cmap='gray')

## Task: find u,v by hand

Try to decompose this image as a product of two vectors (outer product). Choose the vectors 'by hand'.

Example code is provided. What is your error?

> You think you can beat power-iteration (rank-1 SVD)?

In [ ]:
def error_between_images(A,B):
  '''This is often called a Frobenius matrix norm. In the context of images
  it sums up the squared differences between pixel intensities
  (and then takes the sqrt)

  So it's just the Euclidean distance between
  the corresponding long vectors.
  '''
  return np.linalg.norm((A-B).ravel())

In [ ]:
u = np.linspace(0,499,500)
v = np.linspace(0,499,500)

u = (u > 0) & (u < 300) # horizontal
v = (v > 300) & (v < 500) # vertical

my_im_approx = np.outer(v,u)

plt.imshow(my_im_approx, cmap='gray')
plt.title(f'err: {error_between_images(my_im_approx, im_bw)}');

In [ ]:
# just some helper functions

def rank_k_approx(U, S, Vt, k):
  '''
  Gives us rank-k approximation given SVD.

  The reason we don't compute SVD here, is because
  we prefer to pre-compute it once for some image.
  '''
  return U[:, :k] @ S[:k,:k] @ Vt[:k]

def only_k_approx(U, S, Vt, k):
  return np.outer(U.T[k], Vt[k]) * S[k,k]

In [ ]:
U, S, Vt = nice_general_svd(im_bw)

In [ ]:
im1 = rank_k_approx(U, S, Vt, 1)

In [ ]:
print(error_between_images(im1, im_bw))

In [ ]:
assert False
plt.imshow(im1, cmap='gray')

# Constructing the 'layers'

### We are back to the grayscale image!

We'd like to think that SVD decomposes the image into layers, that **add up more and more details**. Here is a simple code which does that.

We can sum up all the layers to get the image back. Or sum just $k$ of them, to get a rank-k approximation.

> Yes, it has a loop. No, it doesn't really matter, since only_k_approx is rather costly.

In [ ]:
def get_SVD_layers(M, max_k):
  ''' Computes max_k images/layers whose
  sum approximates M'''
  U, S, Vt = nice_general_svd(M)
  return np.array([only_k_approx(U, S, Vt, i) for i in range(max_k)])

In [ ]:
plt.imshow(im1, cmap='gray')

In [ ]:
Ls = get_SVD_layers(im, 4)

In [ ]:
Ls.shape

In [ ]:
plot_images_in_a_row(Ls)

In [ ]:
plot_images_in_a_row(np.cumsum(Ls, axis=0))

In [ ]:
plot_images_in_a_row(np.abs(im-np.cumsum(Ls, axis=0)))

# 2) *Singular values*

### Remember the singular values are nonnegative are sorted from highest (dominant) to lowest

## Task: how many singular values would you keep?

In [ ]:
U,S,Vt = nice_general_svd(im)
plt.plot(np.diagonal(S)) # plot the diagonal entries which are the singular values
plt.axvline(x=100, alpha=0.2)
plt.grid()
plt.xlabel('index singular value: $i$');
plt.ylabel(r'the value $\sigma_i$');

In [ ]:
plt.semilogy(np.diagonal(S)) # plot the diagonal entries which are the singular values
plt.axvline(x=100, alpha=0.2)
plt.grid()
plt.xlabel('index singular value: $i$');
plt.ylabel(r'the value $\sigma_i$');

# 3) Low rank compression



## Tasks: low rank image approximation

Play with at animation

> The slider controls parameter k (rank of the resulting matrix)

> What other interesting information can you display? For example: what is the change between k and k+1? Modify the function, if you want.


In [ ]:
sl = lambda: widgets.IntSlider(value=1, min=1, max=len(im)-1)

U, S, Vt = nice_general_svd(im)

def lra(k):
  Ak = rank_k_approx(U, S, Vt, k)
  #plt.clf()
  #plt.figure(figsize = (10,10))
  #plt.imshow(Ak, cmap='magma', vmax = 1, vmin = 0)

  plt.title(f"err = {round(error_between_images(Ak, im), 3)} @ rank {k}")
  plt.imshow(Ak, cmap='gray', vmax=1, vmin=0)

  plt.show()
  #diff = np.abs(Ak-rank_k_approx(U,S,Vt, k-1))
  #plt.imshow(diff, cmap='gray', vmax=1, vmin=-1)

  #Hor = np.outer(U.T[k-1] * S[k-1,k-1], -np.ones(len(Vt))/np.sqrt(len(Vt)) )
  #Vert = np.outer(-np.ones(len(U))/np.sqrt(len(U)), Vt[k-1] * S[k-1,k-1])

  plt.show()

interactive_plot = widgets.interactive(lra, k = sl())

interactive_plot.layout.height = '800px'
interactive_plot.layout.width = '800px'
widgets.HBox([interactive_plot])

# Back to the lecture: optimality

We briefly discuss the Eckhart-Young theorem. It pretty much tells you that -- in a certain sense -- you can't do better than using SVD!

> This also justifies the algorithm I mentioned in the slides -- which iteratively applies rank-1-approximation which we can easily get using **power-iteration**!

## Questions:
How does the error change as we change the approximation rank $k$? How does it compare with the singular value graph we looked at before?

You can also compare the error of *the best* rank-1 approximation with what you came up with by hand for the black and white image!

# 4) Extra: error curve

> Do prefer singular values plot in HW5!


In [ ]:
def error_curve(M):
  U,S,Vt = nice_general_svd(M)
  max_k = np.linalg.matrix_rank(S)

  errors = []
  A = np.zeros_like(M)

  for k in range(1, max_k+1):
    A += only_k_approx(U, S, Vt, k-1)
    errors.append(np.linalg.norm((A-M).ravel()))

  return errors

In [ ]:
errs = error_curve(im)
plt.plot(errs)
plt.grid()